# 1.1 Initialise PySpark

In [ ]:
# import findspark
import findspark
# initialize findspark with spark directory
findspark.init()
# import pyspark
import pyspark
# create spark contextp
sc = pyspark.SparkContext()
# create spark session
spark = pyspark.sql.SparkSession(sc)

In [2]:
# inspect spark session
spark

# 1.2 Import Packages and Data

In [ ]:
# import required packages
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# import pyspark functionalities
from pyspark.sql.types import NumericType
from pyspark.ml import Pipeline
from pyspark.sql.window import Window
from pyspark.sql import functions as F
from pyspark.ml.stat import Correlation
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer, OneHotEncoder, CountVectorizer
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# turn of warnings
import warnings
warnings.filterwarnings("ignore")

In [ ]:
basetable = spark.read.parquet("../Cleaned_data/inititial_basetable.parquet")
basetable.show(5)

+------+--------------+---------------+--------------------+--------+---------+------------+----------+-----------+------------+-----------+----------+------------------+--------------------+-----------------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+------------------+----------------+------------------+------------------+--------------------+---------+--------------------+-------------------+-------+---------+---------------------+------------------------+-----------------------+--------+-------------------+-------------------------+-------------+--------------------+---------------------+---------------------+----------------------+--------------+------------------------+-------------------------+------------------------------+---------------------------+----------------------------------+---------------------------------+--------------------+------------------+------------------+---------+
|ticker|biw

# 2. Modeling

### 2.1 Time-based split

In [6]:
# Determine the the cutoff biweekly index
max_index = basetable.agg(F.max("biweekly_index")).collect()[0][0]
cutoff_index = int(max_index * 0.67)

train_df = basetable.filter(F.col("biweekly_index") <= cutoff_index)
test_df  = basetable.filter(F.col("biweekly_index") > cutoff_index)

In [7]:
print("Training data:")
train_df.groupBy('financing_next_period').count().show()
print("Test data:")
test_df.groupBy('financing_next_period').count().show()

Training data:
+---------------------+-----+
|financing_next_period|count|
+---------------------+-----+
|                    1|   17|
|                    0|  320|
+---------------------+-----+

Test data:
+---------------------+-----+
|financing_next_period|count|
+---------------------+-----+
|                    1|   17|
|                    0|  233|
+---------------------+-----+



In [8]:
train_df.agg(F.max("biweekly_index")).show()
test_df.agg(F.min("biweekly_index")).show()

+-------------------+
|max(biweekly_index)|
+-------------------+
|                 30|
+-------------------+

+-------------------+
|min(biweekly_index)|
+-------------------+
|                 31|
+-------------------+



### 2.2 Checking Missing Values

In [9]:
# get all numeric columns
numeric_cols = [c for c, t in train_df.dtypes if isinstance(train_df.schema[c].dataType, NumericType)]

# count null values in each column
null_counts_train = train_df.select([F.count(F.when(F.col(c).isNull() | F.isnan(c), c)).alias(c) for c in numeric_cols])
null_counts_train.show()

null_counts_test = test_df.select([F.count(F.when(F.col(c).isNull() | F.isnan(c), c)).alias(c) for c in numeric_cols])
null_counts_test.show()

+--------------+-----------+------------+-----------+----------+---------+-------------+-----------------------+----+----------------+-----------------+------------------+------------------+------------+----------------+----------------+-----------------+------------------+--------------------+---------+------------+-------+-------+---------+---------------------+------------------------+-----------------------+---------+-------------------------+-------------+------------------+---------------------+---------------------+----------------------+--------------+------------------------+-------------------------+------------------------------+---------------------------+----------------------------------+---------------------------------+-------------+-----------------+-------------+
|biweekly_index|period_open|period_close|period_high|period_low|avg_close|period_return|benchmark_period_return|beta|price_volatility|return_volatility|avg_intraday_range|max_intraday_range|total_volume|avg_d

In [10]:
train_df = train_df.filter(F.col("prev_period_return").isNotNull())
train_df = train_df.fillna(0)

In [11]:
test_df = test_df.fillna(0)

### 2.3 Select Features

In [12]:
train_df.printSchema()

root
 |-- ticker: string (nullable = true)
 |-- biweekly_index: integer (nullable = true)
 |-- biweekly_period: string (nullable = true)
 |-- longname: string (nullable = true)
 |-- industry: string (nullable = true)
 |-- benchmark: string (nullable = true)
 |-- period_start: date (nullable = true)
 |-- period_end: date (nullable = true)
 |-- period_open: double (nullable = false)
 |-- period_close: double (nullable = false)
 |-- period_high: double (nullable = false)
 |-- period_low: double (nullable = false)
 |-- avg_close: double (nullable = false)
 |-- period_return: double (nullable = false)
 |-- benchmark_period_return: double (nullable = false)
 |-- beta: double (nullable = false)
 |-- price_volatility: double (nullable = false)
 |-- return_volatility: double (nullable = false)
 |-- avg_intraday_range: double (nullable = false)
 |-- max_intraday_range: double (nullable = false)
 |-- total_volume: double (nullable = false)
 |-- avg_daily_volume: double (nullable = false)
 |-- max

In [50]:
label_col = "financing_next_period"
cat_cols = ["industry", "maturity", "locations"]
id_cols =  ["ticker", "biweekly_index","biweekly_period", "longname", "benchmark", "period_start", "period_end"]
exclude_cols = set(id_cols + [label_col])

feature_columns = [c for c in train_df.columns if c not in exclude_cols]

numeric_feature_cols = [col for col in feature_columns if col not in cat_cols]

### 2.4 Scale Numeric Features

only scale:
- price/volume/level/volatility based features
- not: returns/ratio’s/dummies/counts

In [51]:
scale_cols = [
    "period_open","period_close","period_high","period_low","avg_close",
    "price_volatility","avg_intraday_range","max_intraday_range",
    "total_volume","avg_daily_volume","max_daily_volume","volume_volatility",
    "avg_volume_up_days","avg_volume_down_days",
    "avg_gap_size","max_gap",
    "avg_prChg",
    "prev_price_volatility","prev_avg_daily_volume","prev_volume_volatility"
]
no_scale_cols = [c for c in numeric_feature_cols if c not in scale_cols]

Standerdize per ticker, since volume change for large company can be very different to one for another company

In [52]:
# Fit stats op TRAIN
agg_exprs = []
for c in scale_cols:
    agg_exprs += [
        F.avg(F.col(c)).alias(f"{c}__mean"),
        F.stddev_samp(F.col(c)).alias(f"{c}__std"),
    ]

stats_df = train_df.groupBy("ticker").agg(*agg_exprs)


def apply_per_ticker_zscore(df, stats_df, scale_cols):
    df = df.join(stats_df, on="ticker", how="left")
    zcols = []

    for c in scale_cols:
        mu = F.col(f"{c}__mean")
        sd = F.col(f"{c}__std")
        z  = f"{c}__z"

        df = df.withColumn(
            z,
            F.when(sd.isNull() | (sd == 0), F.lit(0.0))
             .otherwise((F.col(c) - mu) / sd)
        )
        zcols.append(z)

    # drop stats cols
    drop_cols = [f"{c}__mean" for c in scale_cols] + [f"{c}__std" for c in scale_cols]
    df = df.drop(*drop_cols)

    return df, zcols


train_s, zcols = apply_per_ticker_zscore(train_df, stats_df, scale_cols)
test_s,  _     = apply_per_ticker_zscore(test_df,  stats_df, scale_cols)

In [53]:
zcols

['period_open__z',
 'period_close__z',
 'period_high__z',
 'period_low__z',
 'avg_close__z',
 'price_volatility__z',
 'avg_intraday_range__z',
 'max_intraday_range__z',
 'total_volume__z',
 'avg_daily_volume__z',
 'max_daily_volume__z',
 'volume_volatility__z',
 'avg_volume_up_days__z',
 'avg_volume_down_days__z',
 'avg_gap_size__z',
 'max_gap__z',
 'avg_prChg__z',
 'prev_price_volatility__z',
 'prev_avg_daily_volume__z',
 'prev_volume_volatility__z']

In [54]:
no_scale_cols

['period_return',
 'benchmark_period_return',
 'beta',
 'return_volatility',
 'gap_count',
 'up_days',
 'down_days',
 'has_property_acquisition',
 'has_exploration_results',
 'price_deviation_from_year',
 'has_financing',
 'prev_period_return',
 'prev_gap_count',
 'prev_exploration_results',
 'prev_property_acquisition',
 'current_financing_announcement',
 'prev_financing_announcement',
 'second_prev_financing_announcement',
 'third_prev_financing_announcement',
 'return_change',
 'volatility_change',
 'volume_change']

### 2.5 Encode Categorical Features

In [55]:
# Indexers
industry_indexer = StringIndexer(inputCol="industry", outputCol="industry_idx", handleInvalid="keep")
maturity_indexer = StringIndexer(inputCol="maturity", outputCol="maturity_idx", handleInvalid="keep")

In [56]:
# One-hot encoder
ENC = OneHotEncoder(
    inputCols=["industry_idx", "maturity_idx"],
    outputCols=["industry_ohe", "maturity_ohe"],
    handleInvalid="keep"
)

In [57]:
CV = CountVectorizer(
    inputCol="locations",
    outputCol="locations_vec",
    binary=True,     # True = presence/absence; set False if counts matter
    minDF=2          # ignore super-rare locations (tune)
)

### 2.6 Final feature set

In [58]:
final_feature_cols = zcols + no_scale_cols + ["industry_ohe", "maturity_ohe"] + ["locations_vec"]

In [67]:
final_feature_cols_rf =  numeric_feature_cols + ["industry_ohe", "maturity_ohe"] + ["locations_vec"]

In [59]:
final_feature_cols

['period_open__z',
 'period_close__z',
 'period_high__z',
 'period_low__z',
 'avg_close__z',
 'price_volatility__z',
 'avg_intraday_range__z',
 'max_intraday_range__z',
 'total_volume__z',
 'avg_daily_volume__z',
 'max_daily_volume__z',
 'volume_volatility__z',
 'avg_volume_up_days__z',
 'avg_volume_down_days__z',
 'avg_gap_size__z',
 'max_gap__z',
 'avg_prChg__z',
 'prev_price_volatility__z',
 'prev_avg_daily_volume__z',
 'prev_volume_volatility__z',
 'period_return',
 'benchmark_period_return',
 'beta',
 'return_volatility',
 'gap_count',
 'up_days',
 'down_days',
 'has_property_acquisition',
 'has_exploration_results',
 'price_deviation_from_year',
 'has_financing',
 'prev_period_return',
 'prev_gap_count',
 'prev_exploration_results',
 'prev_property_acquisition',
 'current_financing_announcement',
 'prev_financing_announcement',
 'second_prev_financing_announcement',
 'third_prev_financing_announcement',
 'return_change',
 'volatility_change',
 'volume_change',
 'industry_ohe',
 '

### 2.7 Assemble Feature Vector

In [60]:
VA = VectorAssembler(inputCols=final_feature_cols, outputCol="features")

In [68]:
VA_RF = VectorAssembler(inputCols=final_feature_cols_rf, outputCol="features")

### 2.8 Logistic Regression

In [61]:
# define linear regression model
LR = LogisticRegression(featuresCol="features", labelCol=label_col)

# fit model on train
lr_pipeline = Pipeline(stages=[
    industry_indexer,
    maturity_indexer,
    ENC,
    CV,          # <-- new stage
    VA,
    LR
])

# Model training
lr_model = lr_pipeline.fit(train_s)

In [62]:
# predictions
lr_predictions = lr_model.transform(test_s)

In [63]:
lr_predictions.select("period_start", "period_end", "financing_next_period", "prediction").show()

+------------+----------+---------------------+----------+
|period_start|period_end|financing_next_period|prediction|
+------------+----------+---------------------+----------+
|  2025-03-24|2025-04-04|                    0|       0.0|
|  2025-04-07|2025-04-17|                    0|       0.0|
|  2025-04-21|2025-05-02|                    0|       0.0|
|  2025-05-05|2025-05-16|                    0|       0.0|
|  2025-05-20|2025-05-30|                    0|       0.0|
|  2025-06-02|2025-06-13|                    0|       0.0|
|  2025-06-16|2025-06-27|                    0|       0.0|
|  2025-06-30|2025-07-11|                    0|       0.0|
|  2025-07-14|2025-07-25|                    0|       0.0|
|  2025-07-28|2025-08-08|                    0|       0.0|
|  2025-08-11|2025-08-22|                    0|       0.0|
|  2025-08-25|2025-08-29|                    0|       0.0|
|  2025-03-24|2025-04-04|                    0|       0.0|
|  2025-04-07|2025-04-17|                    0|       0.

In [64]:
def calculate_classification_metrics(predictions, label_col, prediction_col):
    # calculate true positives, true negatives, false positives, false negatives
    tp = predictions.filter((F.col(label_col) == 1) & (F.col(prediction_col) == 1)).count()
    tn = predictions.filter((F.col(label_col) == 0) & (F.col(prediction_col) == 0)).count()
    fp = predictions.filter((F.col(label_col) == 0) & (F.col(prediction_col) == 1)).count()
    fn = predictions.filter((F.col(label_col) == 1) & (F.col(prediction_col) == 0)).count()

    # calculate accuracy
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    
    # calculate precision
    precision = tp / (tp + fp) if (tp + fp) != 0 else 0  
    
    # calculate recall
    recall = tp / (tp + fn) if (tp + fn) != 0 else 0.0  
    
    # calculate F1 measure
    f1_measure = 2 * (precision * recall) / (precision + recall) if (precision + recall) != 0 else 0.0  

    # print the metrics
    print(f"Accuracy: {accuracy}")
    print(f"Precision: {precision}")
    print(f"Recall: {recall}")
    print(f"F1 measure: {f1_measure}")
    
    return accuracy, precision, recall, f1_measure

In [65]:
# get metrics
lr_accuracy, lr_precision, lr_recall, lr_f1_measure = calculate_classification_metrics(lr_predictions, 'financing_next_period', 'prediction')

Accuracy: 0.708
Precision: 0.07575757575757576
Recall: 0.29411764705882354
F1 measure: 0.12048192771084337


In [66]:
# get AUC
evaluator = BinaryClassificationEvaluator(labelCol="financing_next_period", 
                                          rawPredictionCol="probability", 
                                          metricName="areaUnderROC")

lr_AUC = evaluator.evaluate(lr_predictions)
print(f"AUC: {lr_AUC:.4f}")

AUC: 0.5230


### 2.9 Random Forest

In [70]:
# define random forest model
RF = RandomForestClassifier(featuresCol="features", labelCol="financing_next_period", numTrees=100)

# fit model on train
rf_pipeline = Pipeline(stages=[
    industry_indexer, 
    maturity_indexer, 
    ENC,  
    CV, 
    VA_RF, 
    RF
])

# model training
rf_model = rf_pipeline.fit(train_df)

In [71]:
# predictions
rf_predictions = rf_model.transform(test_df)

In [72]:
# show predictions
rf_predictions.select("period_start", "period_end", "financing_next_period", "prediction").show()

+------------+----------+---------------------+----------+
|period_start|period_end|financing_next_period|prediction|
+------------+----------+---------------------+----------+
|  2025-03-24|2025-04-04|                    0|       0.0|
|  2025-04-07|2025-04-17|                    0|       0.0|
|  2025-04-21|2025-05-02|                    0|       0.0|
|  2025-05-05|2025-05-16|                    0|       0.0|
|  2025-05-20|2025-05-30|                    0|       0.0|
|  2025-06-02|2025-06-13|                    0|       0.0|
|  2025-06-16|2025-06-27|                    0|       0.0|
|  2025-06-30|2025-07-11|                    0|       0.0|
|  2025-07-14|2025-07-25|                    0|       0.0|
|  2025-07-28|2025-08-08|                    0|       0.0|
|  2025-08-11|2025-08-22|                    0|       0.0|
|  2025-08-25|2025-08-29|                    0|       0.0|
|  2025-03-24|2025-04-04|                    0|       0.0|
|  2025-04-07|2025-04-17|                    0|       0.

In [73]:
# get metrics
rf_accuracy, rf_precision, rf_recall, rf_f1_measure = calculate_classification_metrics(rf_predictions, 'financing_next_period', 'prediction')

Accuracy: 0.932
Precision: 0
Recall: 0.0
F1 measure: 0.0


In [74]:
# get AUC
rf_AUC = evaluator.evaluate(rf_predictions)
print(f"AUC: {rf_AUC:.4f}")

AUC: 0.6859


### 2.10 Gradient-Boosted Tree Classifier

In [76]:
# define boosting model
GBT = GBTClassifier(featuresCol="features", labelCol="financing_next_period")

# fit model on train
gbt_pipeline = Pipeline(stages=[
    industry_indexer, 
    maturity_indexer, 
    ENC,
    CV, 
    VA,
    GBT])

# model training
gbt_model = gbt_pipeline.fit(train_s)

In [77]:
# predictions
gbt_predictions = gbt_model.transform(test_s)

In [78]:
gbt_predictions.select("period_start", "period_end", "financing_next_period", "prediction").show()

+------------+----------+---------------------+----------+
|period_start|period_end|financing_next_period|prediction|
+------------+----------+---------------------+----------+
|  2025-03-24|2025-04-04|                    0|       0.0|
|  2025-04-07|2025-04-17|                    0|       0.0|
|  2025-04-21|2025-05-02|                    0|       0.0|
|  2025-05-05|2025-05-16|                    0|       0.0|
|  2025-05-20|2025-05-30|                    0|       0.0|
|  2025-06-02|2025-06-13|                    0|       0.0|
|  2025-06-16|2025-06-27|                    0|       0.0|
|  2025-06-30|2025-07-11|                    0|       0.0|
|  2025-07-14|2025-07-25|                    0|       0.0|
|  2025-07-28|2025-08-08|                    0|       0.0|
|  2025-08-11|2025-08-22|                    0|       0.0|
|  2025-08-25|2025-08-29|                    0|       0.0|
|  2025-03-24|2025-04-04|                    0|       1.0|
|  2025-04-07|2025-04-17|                    0|       0.

In [80]:
gbt_predictions.filter(F.col("financing_next_period") == 1).count()

17

In [81]:
gbt_accuracy, gbt_precision, gbt_recall, gbt_f1_measure = calculate_classification_metrics(gbt_predictions, 'financing_next_period', 'prediction')

Accuracy: 0.884
Precision: 0.22727272727272727
Recall: 0.29411764705882354
F1 measure: 0.25641025641025644


In [82]:
# get AUC
gbt_AUC = evaluator.evaluate(gbt_predictions)
print(f"AUC: {gbt_AUC:.4f}")

AUC: 0.7013


### Future Steps and Potential Improvements

**Autoregressive models**: A base model where you predict the next_announcement based on lagged values of the target variable.

**Feature engineering**: Use different lags or create additional features that capture long-term trends or anomalies in stock price movements/volume.

**Model Tuning**: Implement cross-validation techniques to fine-tune hyperparameters and improve model performance. (not done)

**Class imbalance handling**: Experiment with sampling or using class weights to address the imbalance in the target variable. (not done)

**Data Aggregation**: Explore the possibility of aggregating data of multiple companies to have more points for training the model. This way, you can add company-specific indicators as features, which may enhance the model's predictive power. Part of the assignment is to look how generalizable your model is across different stocks. Also make sure features are normalized properly across different stocks (such as stock price, volume, etc.) to ensure better generalizability. (done)

# 3. Class Imbalance

In [83]:
counts = (
    train_df
    .groupBy(label_col)
    .count()
    .withColumnRenamed("count", "n")
)

total = counts.agg(F.sum("n")).collect()[0][0]

weights = (
    counts
    .withColumn("weight", total / (2 * F.col("n")))
)

weights.show()

+---------------------+---+------------------+
|financing_next_period|  n|            weight|
+---------------------+---+------------------+
|                    1| 16|           9.96875|
|                    0|303|0.5264026402640264|
+---------------------+---+------------------+



In [84]:
def apply_weights(df):
    return df.join(weights, on="financing_next_period", how="left")

In [85]:
train_w = apply_weights(train_df)
test_w  = apply_weights(test_df)


In [86]:
train_w.show()

+---------------------+------+--------------+---------------+--------------------+--------+---------+------------+----------+-----------+------------+-----------+----------+------------------+--------------------+-----------------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+------------------+----------------+------------------+------------------+--------------------+---------+--------------------+--------------------+-------+---------+------------------------+-----------------------+--------+--------------------+-------------------------+-------------+--------------------+---------------------+---------------------+----------------------+--------------+------------------------+-------------------------+------------------------------+---------------------------+----------------------------------+---------------------------------+--------------------+-------------------+------------------+---------+---+-----

# 3.1 LR

In [87]:
# standardising
train_s_w, zcols_w = apply_per_ticker_zscore(train_w, stats_df, scale_cols)
test_s_w,  _     = apply_per_ticker_zscore(test_w,  stats_df, scale_cols)

In [88]:
# define linear regression model
LR = LogisticRegression(featuresCol="features", labelCol=label_col, weightCol="weight")

# fit model on train
lr_pipeline = Pipeline(stages=[
    industry_indexer, 
    maturity_indexer, 
    ENC,
    CV,
    VA, 
    LR
])

# Model training
lr_model = lr_pipeline.fit(train_s_w)

In [89]:
lr_predictions = lr_model.transform(test_s_w)

In [90]:
lr_predictions.select("period_start", "period_end", "financing_next_period", "prediction").show()

+------------+----------+---------------------+----------+
|period_start|period_end|financing_next_period|prediction|
+------------+----------+---------------------+----------+
|  2025-03-24|2025-04-04|                    0|       1.0|
|  2025-04-07|2025-04-17|                    0|       0.0|
|  2025-04-21|2025-05-02|                    0|       0.0|
|  2025-05-05|2025-05-16|                    0|       0.0|
|  2025-05-20|2025-05-30|                    0|       0.0|
|  2025-06-02|2025-06-13|                    0|       0.0|
|  2025-06-16|2025-06-27|                    0|       0.0|
|  2025-06-30|2025-07-11|                    0|       0.0|
|  2025-07-14|2025-07-25|                    0|       0.0|
|  2025-07-28|2025-08-08|                    0|       1.0|
|  2025-08-11|2025-08-22|                    0|       0.0|
|  2025-08-25|2025-08-29|                    0|       0.0|
|  2025-03-24|2025-04-04|                    0|       0.0|
|  2025-04-07|2025-04-17|                    0|       0.

In [91]:
lr_accuracy, lr_precision, lr_recall, lr_f1_measure = calculate_classification_metrics(lr_predictions, 'financing_next_period', 'prediction')

Accuracy: 0.72
Precision: 0.07936507936507936
Recall: 0.29411764705882354
F1 measure: 0.125


In [92]:
# get AUC
evaluator = BinaryClassificationEvaluator(labelCol="financing_next_period", 
                                          rawPredictionCol="probability", 
                                          metricName="areaUnderROC")

lr_AUC = evaluator.evaluate(lr_predictions)
print(f"AUC: {lr_AUC:.4f}")

AUC: 0.5220


### 3.2 RF

In [93]:
# define random forest model
RF = RandomForestClassifier(featuresCol="features", labelCol="financing_next_period", numTrees=100, weightCol="weight")

# fit model on train
rf_pipeline = Pipeline(stages=[
    industry_indexer, 
    maturity_indexer, 
    ENC, 
    CV,
    VA_RF, 
    RF
])

# model training
rf_model = rf_pipeline.fit(train_w)

In [94]:
rf_predictions = rf_model.transform(test_w)

In [95]:
rf_predictions.select("period_start", "period_end", "financing_next_period", "prediction").show()

+------------+----------+---------------------+----------+
|period_start|period_end|financing_next_period|prediction|
+------------+----------+---------------------+----------+
|  2025-03-24|2025-04-04|                    0|       0.0|
|  2025-04-07|2025-04-17|                    0|       0.0|
|  2025-04-21|2025-05-02|                    0|       0.0|
|  2025-05-05|2025-05-16|                    0|       0.0|
|  2025-05-20|2025-05-30|                    0|       0.0|
|  2025-06-02|2025-06-13|                    0|       0.0|
|  2025-06-16|2025-06-27|                    0|       0.0|
|  2025-06-30|2025-07-11|                    0|       0.0|
|  2025-07-14|2025-07-25|                    0|       0.0|
|  2025-07-28|2025-08-08|                    0|       0.0|
|  2025-08-11|2025-08-22|                    0|       0.0|
|  2025-08-25|2025-08-29|                    0|       0.0|
|  2025-03-24|2025-04-04|                    0|       0.0|
|  2025-04-07|2025-04-17|                    0|       0.

In [96]:
rf_accuracy, rf_precision, rf_recall, rf_f1_measure = calculate_classification_metrics(rf_predictions, 'financing_next_period', 'prediction')

Accuracy: 0.932
Precision: 0
Recall: 0.0
F1 measure: 0.0


In [97]:
# get AUC
rf_AUC = evaluator.evaluate(rf_predictions)
print(f"AUC: {rf_AUC:.4f}")

AUC: 0.7407


# 3.3 GBT

In [98]:
# define boosting model
GBT = GBTClassifier(featuresCol="features", labelCol="financing_next_period", weightCol="weight")

# fit model on train
gbt_pipeline = Pipeline(stages=[
    industry_indexer, 
    maturity_indexer, 
    ENC,
    CV, 
    VA, 
    GBT
])

# model training
gbt_model = gbt_pipeline.fit(train_s_w)

In [99]:
gbt_predictions = gbt_model.transform(test_s_w)

In [100]:
gbt_predictions.select("period_start", "period_end", "financing_next_period", "prediction").show()

+------------+----------+---------------------+----------+
|period_start|period_end|financing_next_period|prediction|
+------------+----------+---------------------+----------+
|  2025-03-24|2025-04-04|                    0|       0.0|
|  2025-04-07|2025-04-17|                    0|       0.0|
|  2025-04-21|2025-05-02|                    0|       0.0|
|  2025-05-05|2025-05-16|                    0|       0.0|
|  2025-05-20|2025-05-30|                    0|       0.0|
|  2025-06-02|2025-06-13|                    0|       0.0|
|  2025-06-16|2025-06-27|                    0|       1.0|
|  2025-06-30|2025-07-11|                    0|       0.0|
|  2025-07-14|2025-07-25|                    0|       0.0|
|  2025-07-28|2025-08-08|                    0|       1.0|
|  2025-08-11|2025-08-22|                    0|       0.0|
|  2025-08-25|2025-08-29|                    0|       0.0|
|  2025-03-24|2025-04-04|                    0|       0.0|
|  2025-04-07|2025-04-17|                    0|       0.

In [101]:
gbt_accuracy, gbt_precision, gbt_recall, gbt_f1_measure = calculate_classification_metrics(gbt_predictions, 'financing_next_period', 'prediction')

Accuracy: 0.884
Precision: 0.125
Recall: 0.11764705882352941
F1 measure: 0.12121212121212122


-----------

Sanity Check

In [152]:
from pyspark.sql.functions import rand

shuffled = train_s_w.withColumn("financing_next_period", F.when(rand() > 0.5, 1).otherwise(0))
model_shuffled = lr_pipeline.fit(shuffled)
pred_shuffled = model_shuffled.transform(test_s_w)

In [153]:
shuffled_accuracy, shuffled_precision, shuffled_recall, shuffled_f1_measure = calculate_classification_metrics(pred_shuffled, 'financing_next_period', 'prediction')

Accuracy: 0.372
Precision: 0.045454545454545456
Recall: 0.4117647058823529
F1 measure: 0.08187134502923976


In [156]:
train_pred = lr_model.transform(train_s_w)
test_pred  = lr_model.transform(test_s_w)

calculate_classification_metrics(
    predictions=train_pred,
    prediction_col="prediction",
    label_col="financing_next_period"
)

calculate_classification_metrics(
    predictions=test_pred,
    prediction_col="prediction",
    label_col="financing_next_period"
)


Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1 measure: 1.0
Accuracy: 0.996
Precision: 0.9444444444444444
Recall: 1.0
F1 measure: 0.9714285714285714


(0.996, 0.9444444444444444, 1.0, 0.9714285714285714)

In [198]:
spark.version

'4.0.1'

In [197]:
spark.version
GBTClassifier().hasParam("weightCol")

True